In [ ]:
%load_ext autoreload
%autoreload 2
    
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdate
import datetime as dt
from tqdm import tqdm
from glob import glob
import re
from pathlib import Path

try: 
    import dunestyle.matplotlib as dunestyle
    from cycler import cycler
    plt.rcParams['axes.facecolor'] = '#ffffff'
    plt.rcParams['figure.facecolor'] = '#ffffff'
    plt.rcParams['savefig.facecolor'] = '#ffffff'
    plt.rcParams.update(
        {
        "axes.prop_cycle":cycler(color=['#1f77b4',
                                        '#ff7f0e',
                                        '#2ca02c',
                                        '#d62728',
                                        '#9467bd',
                                        '#8c564b',
                                        '#e377c2',
                                        '#7f7f7f',
                                        '#bcbd22',
                                        '#17becf'])
        })
        
except Exception as e:
    print("Using mplhep... ", e)
    import mplhep
    mplhep.style.use(mplhep.style.ROOT)
    plt.rcParams.update({
                         #'font.size': 23,
                         'grid.linestyle': '--',
                         'axes.grid': True,
                         'figure.autolayout': True,
                         'figure.figsize': [14,6],
                        #  'font.family': 'sans-serif',
                        #  'font.sans-serif': ['Helvetica', 'Helvetica Neue', 'Nimbus Sans', 'Liberation Sans', 'Arial']
                         })
import sys
sys.path.insert(1, '../')
from waffles.np02_utils.AutoMap import getModuleName
from utils import PATH_XE_OUTPUTS
from utils_monitoring import load_database, load_fit_results, make_relative_cols, load_injections
from utils_monitoring import plot_vs_time_per_channel, iplot, execute_by_module


In [ ]:
df_injections = load_injections()
df_injections

In [ ]:
analysisname = 'analysis_results-hvieirad'
typeofana = 'convolution'

path_with_analysis = Path(PATH_XE_OUTPUTS)

# automatic
sufix = 'conv' if typeofana == "convolution" else 'deconv'

In [ ]:
xedb = load_database()
xedb.head()

In [ ]:
dfres = load_fit_results(path_to_data=f"{path_with_analysis}/{analysisname}/{typeofana}", sufix=sufix)
dfres.head()

In [ ]:
df = pd.merge(dfres, xedb, on='run', how='inner')
c1 = set(dfres['run'].unique())
c2 = set(xedb['run'].unique())

print("Runs not in the database:", ','.join(list(c1 - c2)))
print("Missing runs:", ','.join(list(c2 - c1)))
df.head()

In [ ]:
df = make_relative_cols(df, "ppm")
df

In [ ]:
plt.figure(figsize=(14,6))
# plot_vs_time_per_channel(df, y='t3[ns]',  module="C3(1)", showhours=True)
plot_vs_time_per_channel(df, y='t3[ns]',  module="C3(1)", xlim=(pd.to_datetime('2025-09-19 8:00'), pd.to_datetime('2025-09-19 16:30')), showhours=True, label="C3(1)")
plot_vs_time_per_channel(df, y='t3[ns]',  module="C3(2)", xlim=(pd.to_datetime('2025-09-19 8:00'), pd.to_datetime('2025-09-19 16:30')), showhours=True, label="C3(2)", df_injections=df_injections)
# Add shaded regions after all modules are plotted
plt.show()



In [ ]:
plt.figure(figsize=(14,6))
# plot_vs_time_per_channel(df, y='t3[ns]',  module="C3(1)", showhours=True)
execute_by_module(df,
                  plot_vs_time_per_channel,
                  modules=["C"],
                  y='t3[ns]',
                  xlim=(pd.to_datetime('2025-09-19 8:00'), pd.to_datetime('2025-09-19 16:30')),
                  showhours=True,
                  autolabel=True,
                  legendoutside=True,
                  dotitle=False,
                  df_injections=df_injections
                 )

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'iframe'

fig = go.Figure()
execute_by_module(df, iplot, fig=fig, y='t3[ns] relative', x='HV', modules=['C8','C7','C3(1)'])

fig.show()

# iplot(None, getDataFrameModule(df, 'C7(2)'), y='t3[ns]')

In [ ]:
df[df['HV'] == 155.7].groupby(["module"]).mean().loc["C1(1)", "t3[ns]"]